# analyse weather data and comparison bteween local data and weather info

In [64]:
import pandas as pd
import datetime as dt
import xarray as xr
import plotly_express as px

In [65]:
# data sources
local_weather_src =  "data/raw/land_management/weather_observations.csv"

In [66]:
# import and preprocess local_weather pattern

local_weather = pd.read_csv(local_weather_src)
local_weather["datetime"] = local_weather["timestamp"].apply(lambda x: dt.datetime.fromtimestamp(x))


def preprocess_local_weather(local_src: str)->pd.Dataframe:
    local_weather = pd.read_csv(local_weather_src)
    local_weather["datetime"] = local_weather["timestamp"].apply(lambda x: dt.datetime.fromtimestamp(x))
    return local_weather
#---
local_weather_df = preprocess_local_weather(local_weather_src)

In [67]:
# load era5
era5_src= "data/raw/weather/processed/era5_processed_2023-02-01-2026-05-28.nc"

era5 = xr.load_dataset(era5_src)
era5

<xarray.Dataset> Size: 87kB
Dimensions:              (time: 1213, latitude: 4, longitude: 4)
Coordinates:
  * time                 (time) datetime64[ns] 10kB 2023-02-01 ... 2026-05-28
  * latitude             (latitude) float32 16B 31.25 31.0 30.75 30.5
  * longitude            (longitude) float32 16B 254.5 254.8 255.0 255.2
Data variables:
    total_precipitation  (time, latitude, longitude) float32 78kB 0.8652 ... 0.0
Attributes:
    last_updated:           2026-06-09 13:31:46.391208+00:00
    valid_time_start:       1940-01-01
    valid_time_stop:        2025-12-31
    valid_time_stop_era5t:  2026-06-02

In [68]:
imerg_src= "data/raw/weather/processed/imerg_processed_2023-02-01-2026-05-28.nc4"

imerg = xr.load_dataset(imerg_src)
imerg

<xarray.Dataset> Size: 397kB
Dimensions:              (time: 973, lat: 10, lon: 10)
Coordinates:
  * time                 (time) datetime64[ns] 8kB 2023-02-01 ... 2025-09-30
  * lat                  (lat) float64 80B 30.35 30.45 30.55 ... 31.15 31.25
  * lon                  (lon) float32 40B -105.6 -105.4 ... -104.8 -104.7
Data variables:
    total_precipitation  (time, lat, lon) float32 389kB 2.355 3.09 ... 0.0 0.0

In [69]:
# prism
prism_src = "data/raw/weather/processed/prism_processed_2023-02-01-2026-05-28.nc"

prism = xr.load_dataset(prism_src)
prism

<xarray.Dataset> Size: 6MB
Dimensions:              (time: 1213, lat: 24, lon: 24)
Coordinates:
  * time                 (time) datetime64[ns] 10kB 2023-02-01 ... 2026-05-28
  * lat                  (lat) float64 192B 31.29 31.25 31.21 ... 30.38 30.33
  * lon                  (lon) float64 192B -105.6 -105.5 ... -104.7 -104.6
Data variables:
    total_precipitation  (time, lat, lon) float64 6MB 0.0 0.0 0.0 ... 0.0 0.0

## comparison prism /era5 /imerg

In [70]:
era5.plot.scatter(x="time", y="total_precipitation")

## combine dataset

combine datasets and add local as well


In [71]:
combined= xr.merge([
    era5.mean(dim=["latitude", "longitude"]).rename({"total_precipitation": "tp_era5"}),
    prism.mean(dim=["lat", "lon"], skipna=True).rename({"total_precipitation": "tp_prism"}),
    imerg.mean(dim=["lat", "lon"], skipna=True).rename({"total_precipitation": "tp_imerg"})
], join="outer")
combined

<xarray.Dataset> Size: 29kB
Dimensions:   (time: 1213)
Coordinates:
  * time      (time) datetime64[ns] 10kB 2023-02-01 2023-02-02 ... 2026-05-28
Data variables:
    tp_era5   (time) float32 5kB 1.761 2.986 -4.47e-05 ... 2.956 0.005743 0.0
    tp_prism  (time) float64 10kB 0.2096 3.671 0.02903 0.0 ... 11.97 1.816 0.0
    tp_imerg  (time) float32 5kB 2.884 1.032 0.0 0.0 0.0 ... nan nan nan nan nan
Attributes:
    last_updated:           2026-06-09 13:31:46.391208+00:00
    valid_time_start:       1940-01-01
    valid_time_stop:        2025-12-31
    valid_time_stop_era5t:  2026-06-02

In [72]:
# ADD LOCAL 

var_name = "local_day_precip_accumulation"
local_weather["time"] = local_weather["datetime"].apply(lambda x: x.date())
local_weather[var_name]= local_weather[var_name]
# local_weather.set_index("time", inplace=True)
local_weather_trans  =local_weather[["time",var_name ]].groupby("time").sum()
local = xr.Dataset().from_dataframe(local_weather_trans).rename({var_name:"tp_local"})

local

<xarray.Dataset> Size: 14kB
Dimensions:   (time: 898)
Coordinates:
  * time      (time) object 7kB 2023-02-19 2023-02-20 ... 2026-03-25 2026-03-26
Data variables:
    tp_local  (time) float64 7kB 0.2036 0.0 0.03436 0.02419 ... 0.0 0.0 0.0 0.0

In [73]:
local

<xarray.Dataset> Size: 14kB
Dimensions:   (time: 898)
Coordinates:
  * time      (time) object 7kB 2023-02-19 2023-02-20 ... 2026-03-25 2026-03-26
Data variables:
    tp_local  (time) float64 7kB 0.2036 0.0 0.03436 0.02419 ... 0.0 0.0 0.0 0.0

In [74]:
combined= xr.merge([
    combined,
    local
]

)
combined.fillna(0)

/tmp/ipykernel_356561/4158720538.py:1: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'time' ('time',) The recommendation is to set join explicitly for this case.
  combined= xr.merge([


<xarray.Dataset> Size: 68kB
Dimensions:   (time: 2111)
Coordinates:
  * time      (time) object 17kB 2023-02-01 2023-02-02 ... 2026-03-25 2026-03-26
Data variables:
    tp_era5   (time) float32 8kB 1.761 2.986 -4.47e-05 ... 0.0 0.0 4.47e-05
    tp_prism  (time) float64 17kB 0.2096 3.671 0.02903 0.0 ... 0.0 0.0 0.0 0.0
    tp_imerg  (time) float32 8kB 2.884 1.032 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0 0.0
    tp_local  (time) float64 17kB 0.0 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0 0.0
Attributes:
    last_updated:           2026-06-09 13:31:46.391208+00:00
    valid_time_start:       1940-01-01
    valid_time_stop:        2025-12-31
    valid_time_stop_era5t:  2026-06-02

In [75]:
import plotly.graph_objects as go

fig = go.Figure()

for var in ["tp_prism", "tp_era5", "tp_imerg", "tp_local"]:
    fig.add_trace(go.Scatter(
        x=combined["time"].values,
        y=combined[var].values,
        mode="lines",
        name=var,
        line=dict(width=0.5),
    ))

fig.update_layout(xaxis_title="Time", yaxis_title="Total Precipitation")
fig.show()

In [76]:
combined["tp_era5"].values.shape

(2111,)

In [ ]:
from scipy import stats
for var in ["tp_prism", "tp_era5", "tp_imerg"]:
    df = combined[[var, "tp_local"]].to_dataframe().dropna()

    r, p = stats.pearsonr(df[var], df["tp_local"])
    r_squared = r**2
    print(f"correlation between local data and {var}: {r_squared}")

correlation between local data and tp_prism: 0.10386156396130247
correlation between local data and tp_era5: 0.08382567711577991
correlation between local data and tp_imerg: 0.07202771431302468


It seems era5 is the most reliable dataset, however there is notable difference between the local data and the ERA5 dataset. Herebelow some anaylsis

In [112]:
# Exploring era5 dataset vs local sensor for precipitation
sel_var = "tp_prism"
sel_name= sel_var[3:]
ds = combined[[sel_var, "tp_local"]]
ds

<xarray.Dataset> Size: 51kB
Dimensions:   (time: 2111)
Coordinates:
  * time      (time) object 17kB 2023-02-01 2023-02-02 ... 2026-03-25 2026-03-26
Data variables:
    tp_prism  (time) float64 17kB 0.2096 3.671 0.02903 0.0 ... 0.0 0.0 0.0 0.0
    tp_local  (time) float64 17kB nan nan nan nan nan ... 0.0 0.0 0.0 0.0 0.0
Attributes:
    last_updated:           2026-06-09 13:31:46.391208+00:00
    valid_time_start:       1940-01-01
    valid_time_stop:        2025-12-31
    valid_time_stop_era5t:  2026-06-02

In [113]:
# calc percentile
ds[f"perc_{sel_name}"]=100*ds[sel_var]/ds[sel_var].max()
ds["perc_local"]=100*ds["tp_local"]/ds["tp_local"].max()


In [114]:
fig = go.Figure()
fig.add_trace(go.Histogram(x=ds[sel_var], name=sel_var[3:], nbinsx=50, opacity=0.6))
fig.add_trace(go.Histogram(x=ds["tp_local"], name="local", nbinsx=50, opacity=0.6))
 # overlay for side-by-side
fig.update_layout(
    barmode='overlay',
    yaxis_type="log"  # Set y-axis to log scale
)

fig.show()


In [115]:
fig = go.Figure()
fig.add_trace(go.Box(y=ds[sel_var], name=sel_var[3:],  opacity=0.6))
fig.add_trace(go.Box(y=ds["tp_local"], name="local", opacity=0.6))
 # overlay for side-by-side
fig.update_layout(
    barmode='overlay',
)

fig.show()

In [116]:
fig = go.Figure()
fig.add_trace(go.Histogram(x=ds[f"perc_{sel_name}"], name=sel_name, nbinsx=50, opacity=0.6))
fig.add_trace(go.Histogram(x=ds["perc_local"], name="local", nbinsx=50, opacity=0.6))
 # overlay for side-by-side
fig.update_layout(
    barmode='overlay',
    yaxis_type="log"  # Set y-axis to log scale
)

fig.show()

In [117]:
ds

<xarray.Dataset> Size: 84kB
Dimensions:     (time: 2111)
Coordinates:
  * time        (time) object 17kB 2023-02-01 2023-02-02 ... 2026-03-26
Data variables:
    tp_prism    (time) float64 17kB 0.2096 3.671 0.02903 0.0 ... 0.0 0.0 0.0 0.0
    tp_local    (time) float64 17kB nan nan nan nan nan ... 0.0 0.0 0.0 0.0 0.0
    perc_prism  (time) float64 17kB 1.172 20.52 0.1623 0.0 ... 0.0 0.0 0.0 0.0
    perc_local  (time) float64 17kB nan nan nan nan nan ... 0.0 0.0 0.0 0.0 0.0
Attributes:
    last_updated:           2026-06-09 13:31:46.391208+00:00
    valid_time_start:       1940-01-01
    valid_time_stop:        2025-12-31
    valid_time_stop_era5t:  2026-06-02

In [120]:

r, p = stats.pearsonr(ds[f"perc_{sel_name}"], ds["perc_local"])
r_squared = r**2
print(f"correlation between local data and {var}: {r_squared}")

correlation between local data and perc_prism: nan


In [122]:
fig = go.Figure()

for var in [f"perc_{sel_name}", "perc_local"]:
    fig.add_trace(go.Scatter(
        x=ds["time"].values,
        y=ds[var].values,
        mode="lines",
        name=var,
        line=dict(width=0.5),
    ))

fig.update_layout(xaxis_title="Time", yaxis_title="Total Precipitation (percentile)")
fig.show()

unfortunately, even the best weather service seems to not capture even the biggest rain events. Therefore for the WASP dashboard, we will rely on the local weather data as that seems way better than all the others